# Import data

In [ ]:
import sys
sys.path.append("home/565/pv3484/aus_substation_electricity")

%cd aus_substation_electricity/

!pwd

In [ ]:
%run /home/565/pv3484/aus_substation_electricity/import_substation.py

In [ ]:
# imports
import glob, re
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib_scalebar.scalebar import ScaleBar
import numpy as np

## Adding lat and lon info

In [ ]:
from geopy.geocoders import Nominatim
import pandas as pd
import time

# Initialize geocoder
geolocator = Nominatim(user_agent="sydney_demand_mapper")

def get_coords(place):
    """Return (lat, lon) for a suburb name, or (None, None) if not found."""
    try:
        loc = geolocator.geocode(f"{place}, New South Wales, Australia")
        if loc:
            return loc.latitude, loc.longitude
    except Exception as e:
        print(f"Geocoding failed for {place}: {e}")
    return None, None

# Apply geocoding to the 'Name' column
latitudes, longitudes = [], []
for suburb in info['Name']:
    lat, lon = get_coords(suburb)
    latitudes.append(lat)
    longitudes.append(lon)
    time.sleep(1)  # polite pause to avoid hitting API limits

info['latitude'] = latitudes
info['longitude'] = longitudes

### Troublehshooting DeeWhy West

In [ ]:
missing = info[info["latitude"].isna() | info["longitude"].isna()]
missing["Name"].unique()
#Dee Why West doesn't exist as a suburb polygon, so will need to change the name to Dee Why

In [ ]:
dee_why_west_lat = -33.73441
dee_why_west_lon = 151.28278
#Found the lat/lon information online

In [ ]:
info.loc[info["Name"] == "Dee Why West", "latitude"] = dee_why_west_lat
info.loc[info["Name"] == "Dee Why West", "longitude"] = dee_why_west_lon
#inputting lat and lon from online into info

# Mapping

In [ ]:
# load weather station CSVs to get only your stations

station_files = [
    f for f in glob.glob("/home/565/pv3484/aus_substation_electricity/data/BOM_NSW_weather_processed_v4/weather_station_*.csv")
    if re.search(r"weather_station_\d{6}\.csv$", f)
]

frames = []
for f in station_files:
    df = pd.read_csv(f, nrows=1)  # only need 1 row to get lat/lon
    df["station"] = f.split("/")[-1].replace(".csv", "")
    frames.append(df)

obs_sample = pd.concat(frames)

In [ ]:
# build geodataframe

gdf_substations = gpd.GeoDataFrame(
    info[["latitude", "longitude"]].drop_duplicates(),
    geometry=gpd.points_from_xy(
        info[["latitude", "longitude"]].drop_duplicates().longitude,
        info[["latitude", "longitude"]].drop_duplicates().latitude
    ),
    crs="EPSG:4326"
)

weather_locs = obs_sample[["lat", "lon"]].drop_duplicates()
gdf_weather = gpd.GeoDataFrame(
    weather_locs,
    geometry=gpd.points_from_xy(weather_locs.lon, weather_locs.lat),
    crs="EPSG:4326"
)

In [ ]:
# plotting
fig, ax = plt.subplots(figsize=(10, 12))
gdf_substations.plot(ax=ax, color="red", marker="o", markersize=20, alpha=0.8, label="Demand Substations")
gdf_weather.plot(ax=ax, color="blue", marker="s", markersize=30, alpha=0.8, label="Weather Stations")

ctx.add_basemap(ax, crs=gdf_substations.crs, source=ctx.providers.CartoDB.Positron)

# Scale bar
lat_sydney = -33.9
metres_per_degree = 111320 * np.cos(np.radians(lat_sydney))
scalebar = ScaleBar(dx=metres_per_degree, units="m", dimension="si-length",
    length_fraction=0.2, location="lower right", box_alpha=0.7,
    font_properties={"size": 9})
ax.add_artist(scalebar)

# North arrow
x, y, arrow_length = 0.95, 0.95, 0.06
ax.annotate("N", xy=(x, y), xytext=(x, y - arrow_length),
    xycoords="axes fraction", textcoords="axes fraction",
    ha="center", va="center", fontsize=14, fontweight="bold",
    arrowprops=dict(arrowstyle="->", color="black", lw=2))

# Labels — no title, bigger text
ax.legend(loc="upper left", fontsize=12)
ax.set_xlabel("Longitude", fontsize=14)
ax.set_ylabel("Latitude", fontsize=14)
ax.tick_params(axis="both", labelsize=12)

plt.tight_layout()
plt.savefig("/home/565/pv3484/aus_substation_electricity/data/figures/thesis/data_methods/substation_station_map_full", dpi=300)
plt.show()